# Multi-Factor Strategy - Combining Carry, Momentum, and Mean Reversion

This notebook demonstrates how combining multiple alpha signals improves risk-adjusted returns through diversification.

**Key Concept**: Different signals capture different market phenomena:
- **Carry**: Exploits term structure (yield curve)
- **Momentum**: Exploits trend persistence
- **Mean Reversion**: Exploits temporary price deviations

When signals are **uncorrelated or negatively correlated**, combining them:
1. Increases breadth (BR in Grinold-Kahn framework)
2. Reduces idiosyncratic risk
3. Improves Sharpe ratio: **IR = IC × √BR**

**Research Foundation**:
- Grinold & Kahn (1999): Fundamental Law of Active Management
- Clarke et al. (2002): Multi-signal combination methods
- Moskowitz et al. (2012): Time-series momentum across asset classes

**Notebook Contents**:
1. Generate synthetic futures data with realistic market patterns
2. Calculate three alpha signals independently
3. Analyze individual signal performance (IC, Sharpe, returns)
4. Compute signal correlation matrix
5. Combine signals using equal-weight and IC-weighted methods
6. Perform attribution analysis by factor
7. Compare single-factor vs multi-factor portfolios
8. Demonstrate Sharpe ratio improvement from diversification

---

## Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from datetime import date, timedelta
from typing import Dict, List, Tuple
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Setup complete!")

---

## Step 1: Generate Synthetic Futures Data

We'll create realistic futures prices with characteristics that make each signal effective:

- **Carry component**: Positive roll yield (backwardation)
- **Momentum component**: Trending price movements
- **Mean reversion component**: Temporary deviations from trend

This mimics real futures markets where all three phenomena coexist.

In [ ]:
def generate_futures_with_factors(n_contracts=8, n_days=252, base_rate=5.0):
    """
    Generate synthetic futures prices with carry, momentum, and mean reversion.
    
    Returns:
        pl.DataFrame with columns: date, contract, price, roll_date, next_price
    """
    contracts = [f'SFR{q}{y}' for y in ['4', '5'] for q in ['H', 'M', 'U', 'Z']]
    contracts = contracts[:n_contracts]
    
    start_date = date(2023, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    
    data = []
    
    for i, contract in enumerate(contracts):
        # Carry component: futures curve structure
        # Front contracts trade at premium (backwardation)
        carry_effect = 0.15 * (1 - i / len(contracts))  # Front > Back
        
        # Momentum component: trending price movements
        # Different contracts have different trend directions
        trend_direction = 1 if i % 2 == 0 else -1
        trend_strength = 0.0005 * trend_direction  # Daily drift
        
        # Mean reversion component: oscillations around trend
        mean_reversion_speed = 0.05  # Speed of reversion
        
        # Generate price path
        base_price = 100 - base_rate - carry_effect
        prices = [base_price]
        deviation = 0.0  # Current deviation from trend
        
        for day in range(1, n_days):
            # Momentum: persistent drift
            momentum_component = trend_strength
            
            # Mean reversion: pull back to trend
            mean_reversion_component = -mean_reversion_speed * deviation
            
            # Random noise
            noise = np.random.normal(0, 0.02)
            
            # Total price change
            price_change = momentum_component + mean_reversion_component + noise
            new_price = prices[-1] + price_change
            
            # Update deviation from trend
            expected_price = base_price + day * trend_strength
            deviation = new_price - expected_price
            
            prices.append(new_price)
        
        # Determine next contract (for carry calculation)
        next_contract_idx = (i + 1) % len(contracts)
        
        # Roll date (quarterly, 3 months ahead)
        roll_date = start_date + timedelta(days=90)
        
        # Add to dataset
        for day_idx, (d, price) in enumerate(zip(dates, prices)):
            # Next contract price (for carry spread)
            next_price_value = price - carry_effect if i < len(contracts) - 1 else np.nan
            
            data.append({
                'date': d,
                'contract': contract,
                'price': price,
                'roll_date': roll_date,
                'next_price': next_price_value
            })
    
    return pl.DataFrame(data)

# Generate data
futures_data = generate_futures_with_factors(n_contracts=8, n_days=252, base_rate=5.0)

print(f"✓ Generated {futures_data['contract'].n_unique()} contracts over {futures_data['date'].n_unique()} days")
print(f"\nContracts: {sorted(futures_data['contract'].unique().to_list())}")
print(f"\nSample data:")
print(futures_data.head(10))

In [ ]:
# Visualize price paths to see the embedded factors
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Price paths (shows momentum)
contracts = sorted(futures_data['contract'].unique().to_list())
for contract in contracts:
    contract_data = futures_data.filter(pl.col('contract') == contract)
    prices_pd = contract_data.select(['date', 'price']).to_pandas()
    axes[0].plot(prices_pd['date'], prices_pd['price'], label=contract, alpha=0.7, linewidth=2)

axes[0].set_title('Futures Price Paths (Momentum Component Visible)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=11)
axes[0].set_ylabel('Price', fontsize=11)
axes[0].legend(ncol=4, loc='best', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Carry structure (front vs back)
# Show calendar spreads over time
first_date = futures_data['date'].min()
sample_dates = futures_data.filter(pl.col('date').is_in([first_date])).to_pandas()

calendar_spreads = []
for _, row in sample_dates.iterrows():
    if not pd.isna(row['next_price']):
        spread = row['price'] - row['next_price']
        calendar_spreads.append({'contract': row['contract'], 'spread': spread})

spread_df = pd.DataFrame(calendar_spreads)
axes[1].barh(spread_df['contract'], spread_df['spread'], color='steelblue', alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title(f'Calendar Spreads (Front - Back) on {first_date} (Carry Component)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Spread (Price Points)', fontsize=11)
axes[1].set_ylabel('Contract', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("  • Top plot: Different contracts show trending behavior (momentum)")
print("  • Bottom plot: Positive spreads indicate backwardation (carry opportunity)")
print("  • Prices oscillate around trends (mean reversion component)")

---

## Step 2: Calculate Individual Signals

We'll generate three independent signals:

1. **Carry Signal**: Calendar spread (front - back) annualized
2. **Momentum Signal**: 60-day price trend
3. **Mean Reversion Signal**: Deviation from 20-day moving average

In [ ]:
from Signals.Futures.CarrySignal import CarrySignal
from Signals.Futures.MomentumSignal import MomentumSignal
from Signals.Futures.MeanReversionSignal import MeanReversionSignal

# Create signal calculators
carry_signal = CarrySignal(name='carry', standardize=True, annualize=True)
momentum_signal = MomentumSignal(name='momentum', lookback_days=60, standardize=True)
mean_reversion_signal = MeanReversionSignal(name='mean_reversion', lookback_days=20, standardize=True)

print("✓ Signal calculators created:")
print(f"  • {carry_signal}")
print(f"  • {momentum_signal}")
print(f"  • {mean_reversion_signal}")

In [ ]:
# Mock market data provider
class MockMarketData:
    """Simple market data provider for our synthetic data."""
    
    def __init__(self, data: pl.DataFrame):
        self.data = data
    
    def get_price_history(self, contract: str, start_date: date, end_date: date) -> pl.DataFrame:
        """Get price history for a contract."""
        return self.data.filter(
            (pl.col('contract') == contract) &
            (pl.col('date') >= start_date) &
            (pl.col('date') <= end_date)
        ).sort('date')

mdp = MockMarketData(futures_data)
print("✓ Mock market data provider initialized")

In [ ]:
# Calculate signals for all contracts and dates
def calculate_signals_over_time(
    signal, 
    contracts: List[str], 
    dates: List[date], 
    mdp, 
    min_history: int = 70
) -> pd.DataFrame:
    """
    Calculate signal values over time for multiple contracts.
    
    Returns:
        DataFrame with dates as index, contracts as columns, signal values as entries
    """
    results = []
    
    for as_of in dates[min_history:]:  # Skip initial dates without enough history
        try:
            signals = signal.calculate(contracts, mdp, as_of)
            row = {'date': as_of}
            row.update(signals)
            results.append(row)
        except Exception as e:
            # Skip dates with errors
            continue
    
    df = pd.DataFrame(results)
    df = df.set_index('date')
    return df

# Calculate all signals
contracts = sorted(futures_data['contract'].unique().to_list())
dates = sorted(futures_data['date'].unique().to_list())

print("Calculating signals over time...")
carry_signals_df = calculate_signals_over_time(carry_signal, contracts, dates, mdp, min_history=70)
momentum_signals_df = calculate_signals_over_time(momentum_signal, contracts, dates, mdp, min_history=70)
mean_reversion_signals_df = calculate_signals_over_time(mean_reversion_signal, contracts, dates, mdp, min_history=70)

print(f"\n✓ Signals calculated for {len(carry_signals_df)} dates")
print(f"\nSample carry signals (latest date):")
print(carry_signals_df.iloc[-1])
print(f"\nSample momentum signals (latest date):")
print(momentum_signals_df.iloc[-1])
print(f"\nSample mean reversion signals (latest date):")
print(mean_reversion_signals_df.iloc[-1])

---

## Step 3: Analyze Individual Signal Performance

Calculate Information Coefficient (IC) for each signal to measure forecasting skill.

In [ ]:
# Calculate returns for IC calculation
def calculate_forward_returns(futures_data: pl.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """
    Calculate forward returns for each contract.
    
    Args:
        futures_data: Price data
        horizon: Periods ahead to calculate returns
    
    Returns:
        DataFrame with dates as index, contracts as columns, returns as entries
    """
    contracts = sorted(futures_data['contract'].unique().to_list())
    
    returns_data = []
    for contract in contracts:
        contract_data = futures_data.filter(pl.col('contract') == contract).sort('date')
        prices = contract_data['price'].to_numpy()
        dates_list = contract_data['date'].to_list()
        
        # Calculate forward returns
        for i in range(len(prices) - horizon):
            ret = (prices[i + horizon] - prices[i]) / prices[i]
            returns_data.append({
                'date': dates_list[i],
                'contract': contract,
                'return': ret
            })
    
    returns_df = pd.DataFrame(returns_data)
    returns_pivot = returns_df.pivot(index='date', columns='contract', values='return')
    return returns_pivot

forward_returns = calculate_forward_returns(futures_data, horizon=5)  # 5-day forward returns
print(f"✓ Calculated forward returns for {len(forward_returns)} dates")
print(f"\nSample forward returns (latest 5 dates):")
print(forward_returns.tail())

In [ ]:
# Calculate IC (Information Coefficient) for each signal
def calculate_ic(signals_df: pd.DataFrame, returns_df: pd.DataFrame) -> Dict[str, float]:
    """
    Calculate Information Coefficient (correlation between signals and returns).
    
    Returns:
        Dict with IC statistics
    """
    # Align dates
    common_dates = signals_df.index.intersection(returns_df.index)
    signals_aligned = signals_df.loc[common_dates]
    returns_aligned = returns_df.loc[common_dates]
    
    # Flatten to 1D arrays for correlation
    signals_flat = signals_aligned.values.flatten()
    returns_flat = returns_aligned.values.flatten()
    
    # Remove NaN values
    mask = ~(np.isnan(signals_flat) | np.isnan(returns_flat))
    signals_clean = signals_flat[mask]
    returns_clean = returns_flat[mask]
    
    if len(signals_clean) == 0:
        return {'ic': 0.0, 'p_value': 1.0, 'n': 0}
    
    # Calculate Pearson correlation
    ic, p_value = stats.pearsonr(signals_clean, returns_clean)
    
    return {
        'ic': ic,
        'p_value': p_value,
        'n': len(signals_clean)
    }

# Calculate IC for each signal
carry_ic = calculate_ic(carry_signals_df, forward_returns)
momentum_ic = calculate_ic(momentum_signals_df, forward_returns)
mean_reversion_ic = calculate_ic(mean_reversion_signals_df, forward_returns)

print("Individual Signal Performance (IC Analysis):")
print("=" * 60)
print(f"\n1. Carry Signal:")
print(f"   IC: {carry_ic['ic']:>8.4f}")
print(f"   p-value: {carry_ic['p_value']:>8.4f}")
print(f"   Significance: {'***' if carry_ic['p_value'] < 0.01 else '**' if carry_ic['p_value'] < 0.05 else '*' if carry_ic['p_value'] < 0.10 else 'not significant'}")

print(f"\n2. Momentum Signal:")
print(f"   IC: {momentum_ic['ic']:>8.4f}")
print(f"   p-value: {momentum_ic['p_value']:>8.4f}")
print(f"   Significance: {'***' if momentum_ic['p_value'] < 0.01 else '**' if momentum_ic['p_value'] < 0.05 else '*' if momentum_ic['p_value'] < 0.10 else 'not significant'}")

print(f"\n3. Mean Reversion Signal:")
print(f"   IC: {mean_reversion_ic['ic']:>8.4f}")
print(f"   p-value: {mean_reversion_ic['p_value']:>8.4f}")
print(f"   Significance: {'***' if mean_reversion_ic['p_value'] < 0.01 else '**' if mean_reversion_ic['p_value'] < 0.05 else '*' if mean_reversion_ic['p_value'] < 0.10 else 'not significant'}")

print("\n📊 IC Interpretation:")
print("   • IC > 0.05: Good forecasting skill")
print("   • IC > 0.10: Very good forecasting skill")
print("   • IC > 0.15: Exceptional (rare)")
print("   • p-value < 0.05: Statistically significant")

---

## Step 4: Signal Correlation Analysis

Key insight: **Low correlation between signals = higher diversification benefit**

In [ ]:
# Combine all signals into one DataFrame for correlation analysis
def align_signals(*signal_dfs):
    """
    Align multiple signal DataFrames to common dates and contracts.
    """
    # Find common dates
    common_dates = signal_dfs[0].index
    for df in signal_dfs[1:]:
        common_dates = common_dates.intersection(df.index)
    
    # Find common contracts
    common_contracts = signal_dfs[0].columns
    for df in signal_dfs[1:]:
        common_contracts = common_contracts.intersection(df.columns)
    
    # Return aligned DataFrames
    return [df.loc[common_dates, common_contracts] for df in signal_dfs]

# Align signals
carry_aligned, momentum_aligned, mr_aligned = align_signals(
    carry_signals_df, 
    momentum_signals_df, 
    mean_reversion_signals_df
)

# Flatten signals for correlation calculation
carry_flat = carry_aligned.values.flatten()
momentum_flat = momentum_aligned.values.flatten()
mr_flat = mr_aligned.values.flatten()

# Remove NaN values
mask = ~(np.isnan(carry_flat) | np.isnan(momentum_flat) | np.isnan(mr_flat))
carry_clean = carry_flat[mask]
momentum_clean = momentum_flat[mask]
mr_clean = mr_flat[mask]

# Create correlation matrix
signal_matrix = np.column_stack([carry_clean, momentum_clean, mr_clean])
correlation_matrix = np.corrcoef(signal_matrix.T)

corr_df = pd.DataFrame(
    correlation_matrix,
    index=['Carry', 'Momentum', 'Mean Reversion'],
    columns=['Carry', 'Momentum', 'Mean Reversion']
)

print("Signal Correlation Matrix:")
print("=" * 50)
print(corr_df.round(3))
print("\n💡 Key Observations:")
print(f"   • Carry vs Momentum: {corr_df.loc['Carry', 'Momentum']:.3f}")
print(f"   • Carry vs Mean Reversion: {corr_df.loc['Carry', 'Mean Reversion']:.3f}")
print(f"   • Momentum vs Mean Reversion: {corr_df.loc['Momentum', 'Mean Reversion']:.3f}")
print("\nLow correlations = good diversification potential!")

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
            vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Correlation'})
plt.title('Signal Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  • Green = positive correlation (signals agree)")
print("  • Red = negative correlation (signals disagree)")
print("  • Yellow/white = low correlation (independent signals)")
print("\nIdeal for multi-factor: Low/negative correlations between factors")

---

## Step 5: Signal Combination Methods

Compare two approaches:

1. **Equal Weight**: Simple average (baseline)
2. **IC-Weighted**: Weight by forecasting skill (better when ICs known)

In [ ]:
from Signals.SignalCombiner import SignalCombiner

# Create signal combiner
combiner = SignalCombiner()

# Prepare signals dict format for combiner
def prepare_signals_dict(carry_df, momentum_df, mr_df, as_of):
    """Convert DataFrames to dict format for SignalCombiner."""
    return {
        'carry': carry_df.loc[as_of].to_dict(),
        'momentum': momentum_df.loc[as_of].to_dict(),
        'mean_reversion': mr_df.loc[as_of].to_dict()
    }

# Calculate combined signals for all dates
equal_weight_signals = []
ic_weighted_signals = []

# IC estimates for weighting
ic_estimates = {
    'carry': abs(carry_ic['ic']),
    'momentum': abs(momentum_ic['ic']),
    'mean_reversion': abs(mean_reversion_ic['ic'])
}

print("Combining signals using different methods...")
print(f"\nIC estimates for weighting:")
for signal_name, ic_val in ic_estimates.items():
    print(f"  {signal_name}: {ic_val:.4f}")

for as_of in carry_aligned.index:
    signals_dict = prepare_signals_dict(carry_aligned, momentum_aligned, mr_aligned, as_of)
    
    # Equal weight
    equal_combined = combiner.combine(signals_dict, method='equal')
    equal_weight_signals.append({'date': as_of, **equal_combined})
    
    # IC-weighted
    ic_combined = combiner.combine(signals_dict, method='ic_weighted', ic_estimates=ic_estimates)
    ic_weighted_signals.append({'date': as_of, **ic_combined})

# Convert to DataFrames
equal_weight_df = pd.DataFrame(equal_weight_signals).set_index('date')
ic_weighted_df = pd.DataFrame(ic_weighted_signals).set_index('date')

print(f"\n✓ Combined signals calculated for {len(equal_weight_df)} dates")
print(f"\nSample equal-weight combined signals (latest):")
print(equal_weight_df.iloc[-1])
print(f"\nSample IC-weighted combined signals (latest):")
print(ic_weighted_df.iloc[-1])

In [ ]:
# Calculate IC for combined signals
equal_weight_ic = calculate_ic(equal_weight_df, forward_returns)
ic_weighted_ic = calculate_ic(ic_weighted_df, forward_returns)

print("Combined Signal Performance:")
print("=" * 60)
print(f"\n1. Equal Weight Combination:")
print(f"   IC: {equal_weight_ic['ic']:>8.4f}")
print(f"   p-value: {equal_weight_ic['p_value']:>8.4f}")
print(f"   vs Best Individual: {max(carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic']):.4f}")

print(f"\n2. IC-Weighted Combination:")
print(f"   IC: {ic_weighted_ic['ic']:>8.4f}")
print(f"   p-value: {ic_weighted_ic['p_value']:>8.4f}")
print(f"   vs Equal Weight: {equal_weight_ic['ic']:.4f}")

print("\n📊 Summary:")
print(f"   Best individual IC: {max(carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic']):.4f}")
print(f"   Equal weight IC:    {equal_weight_ic['ic']:.4f}")
print(f"   IC-weighted IC:     {ic_weighted_ic['ic']:.4f}")

improvement = (ic_weighted_ic['ic'] - max(carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic'])) / max(carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic']) * 100
print(f"\n💡 IC-weighted improvement over best single factor: {improvement:.1f}%")

---

## Step 6: Portfolio Construction and Backtesting

Build portfolios using:
1. Single-factor strategies (carry-only, momentum-only, mean-reversion-only)
2. Multi-factor strategy (IC-weighted combination)

In [ ]:
from Signals.AlphaGenerator import AlphaGenerator
from Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkage
from Optimizer.MeanVarianceOptimizer import MeanVarianceOptimizer

# Create alpha generator
alpha_gen = AlphaGenerator(IC=0.05)  # Conservative IC assumption

# Calculate returns history for volatility estimation
returns_history_dict = {}
for contract in contracts:
    contract_data = futures_data.filter(pl.col('contract') == contract).sort('date')
    prices = contract_data['price'].to_numpy()
    returns = np.diff(prices) / prices[:-1]
    returns_history_dict[contract] = returns

# Create returns DataFrame (for covariance estimation)
min_length = min(len(r) for r in returns_history_dict.values())
returns_history = pd.DataFrame({
    contract: returns_history_dict[contract][-min_length:]
    for contract in contracts
})

print(f"✓ Returns history: {len(returns_history)} periods")
print(f"\nSample returns (latest 5 periods):")
print(returns_history.tail())

In [ ]:
# Function to run backtest with given signals
def run_strategy_backtest(
    signals_df: pd.DataFrame,
    returns_df: pd.DataFrame,
    risk_aversion: float = 1.0
) -> Dict:
    """
    Run backtest for a signal strategy.
    
    Returns:
        Dict with portfolio_returns, sharpe, total_return
    """
    # Align dates
    common_dates = signals_df.index.intersection(returns_df.index)
    signals_aligned = signals_df.loc[common_dates]
    returns_aligned = returns_df.loc[common_dates]
    
    # Estimate covariance (using all history)
    cov_estimator = LedoitWolfShrinkage()
    cov_estimator.fit(returns_history)
    cov_matrix = pd.DataFrame(
        cov_estimator.cov_matrix_,
        index=contracts,
        columns=contracts
    )
    
    # Create optimizer
    optimizer = MeanVarianceOptimizer(
        risk_aversion=risk_aversion,
        long_only=True
    )
    
    # Run backtest
    portfolio_returns = []
    
    for i in range(len(signals_aligned)):
        # Get signals for this period
        period_signals = signals_aligned.iloc[i]
        
        # Convert to alphas (expected returns)
        signals_dict = period_signals.to_dict()
        alphas = alpha_gen.signals_to_alphas(
            signals_dict,
            returns_history,
            as_of=common_dates[i]
        )
        
        # Optimize portfolio
        alphas_series = pd.Series(alphas)
        weights = optimizer.optimize(alphas_series, cov_matrix)
        
        # Calculate portfolio return
        period_returns = returns_aligned.iloc[i]
        weights_aligned = pd.Series({c: weights.get(c, 0.0) for c in period_returns.index})
        portfolio_return = (weights_aligned * period_returns).sum()
        portfolio_returns.append(portfolio_return)
    
    # Calculate metrics
    portfolio_returns = np.array(portfolio_returns)
    sharpe = np.mean(portfolio_returns) / np.std(portfolio_returns) * np.sqrt(252) if np.std(portfolio_returns) > 0 else 0
    total_return = np.prod(1 + portfolio_returns) - 1
    
    return {
        'portfolio_returns': portfolio_returns,
        'sharpe': sharpe,
        'total_return': total_return,
        'dates': common_dates
    }

print("Running backtests for all strategies...")

# Run backtests
carry_backtest = run_strategy_backtest(carry_aligned, forward_returns)
momentum_backtest = run_strategy_backtest(momentum_aligned, forward_returns)
mr_backtest = run_strategy_backtest(mr_aligned, forward_returns)
equal_weight_backtest = run_strategy_backtest(equal_weight_df, forward_returns)
ic_weighted_backtest = run_strategy_backtest(ic_weighted_df, forward_returns)

print("\n✓ Backtests complete!")

---

## Step 7: Performance Comparison

**Key Question**: Does combining signals improve risk-adjusted returns?

In [ ]:
# Create performance summary table
performance_summary = pd.DataFrame({
    'Strategy': [
        'Carry Only',
        'Momentum Only',
        'Mean Reversion Only',
        'Multi-Factor (Equal Weight)',
        'Multi-Factor (IC-Weighted)'
    ],
    'Sharpe Ratio': [
        carry_backtest['sharpe'],
        momentum_backtest['sharpe'],
        mr_backtest['sharpe'],
        equal_weight_backtest['sharpe'],
        ic_weighted_backtest['sharpe']
    ],
    'Total Return': [
        carry_backtest['total_return'],
        momentum_backtest['total_return'],
        mr_backtest['total_return'],
        equal_weight_backtest['total_return'],
        ic_weighted_backtest['total_return']
    ],
    'IC': [
        carry_ic['ic'],
        momentum_ic['ic'],
        mean_reversion_ic['ic'],
        equal_weight_ic['ic'],
        ic_weighted_ic['ic']
    ]
})

print("Performance Summary:")
print("=" * 80)
print(performance_summary.to_string(index=False))

# Calculate improvement
best_single_sharpe = performance_summary.iloc[:3]['Sharpe Ratio'].max()
multi_factor_sharpe = performance_summary.iloc[-1]['Sharpe Ratio']
sharpe_improvement = (multi_factor_sharpe - best_single_sharpe) / best_single_sharpe * 100

print(f"\n💡 Key Results:")
print(f"   Best single-factor Sharpe: {best_single_sharpe:.3f}")
print(f"   Multi-factor Sharpe:       {multi_factor_sharpe:.3f}")
print(f"   Improvement:               {sharpe_improvement:+.1f}%")
print(f"\n   This demonstrates the power of diversification across uncorrelated signals!")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Sharpe Ratio comparison
strategies = performance_summary['Strategy'].tolist()
sharpes = performance_summary['Sharpe Ratio'].tolist()
colors = ['steelblue', 'steelblue', 'steelblue', 'orange', 'green']

axes[0, 0].barh(strategies, sharpes, color=colors, alpha=0.7)
axes[0, 0].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Sharpe Ratio', fontsize=11)
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Plot 2: Total Return comparison
total_returns = performance_summary['Total Return'].tolist()
axes[0, 1].barh(strategies, total_returns, color=colors, alpha=0.7)
axes[0, 1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Total Return Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Total Return', fontsize=11)
axes[0, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Plot 3: Cumulative returns over time
cum_carry = np.cumprod(1 + carry_backtest['portfolio_returns'])
cum_momentum = np.cumprod(1 + momentum_backtest['portfolio_returns'])
cum_mr = np.cumprod(1 + mr_backtest['portfolio_returns'])
cum_equal = np.cumprod(1 + equal_weight_backtest['portfolio_returns'])
cum_ic = np.cumprod(1 + ic_weighted_backtest['portfolio_returns'])

dates = carry_backtest['dates']
axes[1, 0].plot(dates, cum_carry, label='Carry Only', alpha=0.7, linewidth=2)
axes[1, 0].plot(dates, cum_momentum, label='Momentum Only', alpha=0.7, linewidth=2)
axes[1, 0].plot(dates, cum_mr, label='Mean Reversion Only', alpha=0.7, linewidth=2)
axes[1, 0].plot(dates, cum_equal, label='Multi-Factor (Equal)', alpha=0.9, linewidth=2.5, linestyle='--')
axes[1, 0].plot(dates, cum_ic, label='Multi-Factor (IC-Weighted)', alpha=0.9, linewidth=3, color='green')
axes[1, 0].set_title('Cumulative Returns Over Time', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Date', fontsize=11)
axes[1, 0].set_ylabel('Growth of $1', fontsize=11)
axes[1, 0].legend(loc='best', fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: IC comparison
ics = performance_summary['IC'].tolist()
axes[1, 1].barh(strategies, ics, color=colors, alpha=0.7)
axes[1, 1].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Information Coefficient (IC) Comparison', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('IC', fontsize=11)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n📊 Chart Interpretation:")
print("  • Top-left: Sharpe ratio (risk-adjusted returns) - higher is better")
print("  • Top-right: Total return - higher is better")
print("  • Bottom-left: Cumulative wealth - green line (multi-factor) should outperform")
print("  • Bottom-right: IC (forecasting skill) - higher is better")

---

## Step 8: Performance Attribution by Factor

Decompose multi-factor returns to understand each signal's contribution.

In [ ]:
# Calculate factor contributions using IC-weighted portfolio
# Attribution: weight × (factor return)

# Normalize IC weights
total_ic = sum(ic_estimates.values())
ic_weights = {k: v / total_ic for k, v in ic_estimates.items()}

print("Factor Weights (IC-Weighted):")
print("=" * 50)
for factor, weight in ic_weights.items():
    print(f"  {factor:<20} {weight:>6.2%}")

# Calculate attributed returns
carry_contribution = ic_weights['carry'] * carry_backtest['total_return']
momentum_contribution = ic_weights['momentum'] * momentum_backtest['total_return']
mr_contribution = ic_weights['mean_reversion'] * mr_backtest['total_return']

print("\nFactor Contributions to Total Return:")
print("=" * 50)
print(f"  Carry:           {carry_contribution:>8.2%}")
print(f"  Momentum:        {momentum_contribution:>8.2%}")
print(f"  Mean Reversion:  {mr_contribution:>8.2%}")
print(f"  {'-' * 40}")
print(f"  Total (approx):  {carry_contribution + momentum_contribution + mr_contribution:>8.2%}")
print(f"  Actual:          {ic_weighted_backtest['total_return']:>8.2%}")

# Visualize attribution
fig, ax = plt.subplots(figsize=(10, 6))
contributions = [carry_contribution, momentum_contribution, mr_contribution]
factors = ['Carry', 'Momentum', 'Mean Reversion']
colors_attr = ['steelblue', 'orange', 'green']

ax.barh(factors, contributions, color=colors_attr, alpha=0.7)
ax.set_title('Performance Attribution by Factor', fontsize=14, fontweight='bold')
ax.set_xlabel('Contribution to Total Return', fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---

## Step 9: IC Decomposition Analysis

Understand how combining signals affects forecasting skill.

In [ ]:
# IC decomposition
print("IC Decomposition Analysis:")
print("=" * 60)
print(f"\nIndividual Factor ICs:")
print(f"  Carry:           {carry_ic['ic']:>8.4f}")
print(f"  Momentum:        {momentum_ic['ic']:>8.4f}")
print(f"  Mean Reversion:  {mean_reversion_ic['ic']:>8.4f}")
print(f"  Average:         {np.mean([carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic']]):>8.4f}")

print(f"\nCombined Signal ICs:")
print(f"  Equal Weight:    {equal_weight_ic['ic']:>8.4f}")
print(f"  IC-Weighted:     {ic_weighted_ic['ic']:>8.4f}")

print(f"\n💡 Key Insights:")
print(f"   1. IC improvement from combination: {(ic_weighted_ic['ic'] / np.mean([carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic']]) - 1) * 100:.1f}%")
print(f"   2. IC-weighted > Equal-weight: {ic_weighted_ic['ic'] > equal_weight_ic['ic']}")
print(f"   3. Combination captures complementary signals effectively")

# Visualize IC comparison
ic_comparison = pd.DataFrame({
    'Signal': ['Carry', 'Momentum', 'Mean Reversion', 'Equal Weight', 'IC-Weighted'],
    'IC': [carry_ic['ic'], momentum_ic['ic'], mean_reversion_ic['ic'], 
           equal_weight_ic['ic'], ic_weighted_ic['ic']],
    'Type': ['Single', 'Single', 'Single', 'Combined', 'Combined']
})

fig, ax = plt.subplots(figsize=(12, 6))
colors_ic = ['steelblue' if t == 'Single' else 'green' for t in ic_comparison['Type']]
bars = ax.barh(ic_comparison['Signal'], ic_comparison['IC'], color=colors_ic, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax.set_title('Information Coefficient (IC) Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('IC (Correlation with Forward Returns)', fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', alpha=0.7, label='Single Factor'),
    Patch(facecolor='green', alpha=0.7, label='Combined Factor')
]
ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()

---

## Summary: Key Takeaways

### 1. Multi-Factor Strategy Benefits

✅ **Improved Sharpe Ratio**: Combining uncorrelated signals increases risk-adjusted returns

✅ **Increased Breadth (BR)**: More independent bets → better IR (Information Ratio)

✅ **Diversification**: Reduces idiosyncratic risk from any single factor

### 2. Signal Combination Methods

- **Equal Weight**: Simple, robust baseline
- **IC-Weighted**: Better when you have reliable IC estimates
- **Orthogonalization**: Best when signals are correlated (not shown here)

### 3. Grinold-Kahn Framework

The Fundamental Law of Active Management:

**IR = IC × √BR**

Where:
- **IR**: Information Ratio (Sharpe of active return)
- **IC**: Information Coefficient (forecasting skill)
- **BR**: Breadth (number of independent bets)

Multi-factor strategies improve **BR** by combining uncorrelated signals!

### 4. Practical Implementation

```python
# Create signals
carry = CarrySignal()
momentum = MomentumSignal(lookback_days=60)
mean_reversion = MeanReversionSignal(lookback_days=20)

# Calculate signals
signals = {
    'carry': carry.calculate(instruments, mdp, as_of),
    'momentum': momentum.calculate(instruments, mdp, as_of),
    'mean_reversion': mean_reversion.calculate(instruments, mdp, as_of)
}

# Combine signals (IC-weighted)
combiner = SignalCombiner()
combined = combiner.combine(signals, method='ic_weighted', ic_estimates=ic_dict)

# Convert to alphas and optimize
alpha_gen = AlphaGenerator(IC=0.05)
alphas = alpha_gen.signals_to_alphas(combined, returns_history, as_of)
weights = optimizer.optimize(alphas, covariance_matrix)
```

### 5. Next Steps

To extend this analysis:

1. **Real Data**: Replace synthetic data with actual futures prices
2. **More Factors**: Add value, volatility, curve positioning signals
3. **Regime Analysis**: IC may vary by market regime (trending vs ranging)
4. **Transaction Costs**: Include realistic trading costs
5. **Walk-Forward Testing**: Use rolling window IC estimation

See `docs/SIGNAL_COMBINATION_METHODS.md` for detailed combination strategies.